# FedTFT Sensitivity Preprocessing Notebook
## Purpose
Run the full preprocessing pipeline with different config values to produce
variant-specific `last_npy_data/` datasets for sensitivity analysis.

| Experiment | Config change | Output tag |
|---|---|---|
| Baseline | σ=0.5, cosinor 48h/168h, seq_len=192 | `baseline` |
| Aug σ=0.25 | σ=0.25 | `aug_sigma025` |
| Aug σ=1.0 | σ=1.0 | `aug_sigma100` |
| Cosinor 24h/120h | short=24, long=120 | `cosinor_24_120` |
| Cosinor 72h/192h | short=72, long=192 | `cosinor_72_192` |
| Seq 96 steps (24h) | seq_len=96 | `seqlen_096` |
| Seq 288 steps (72h) | seq_len=288 | `seqlen_288` |

**After running each cell pair below, point FL training at the variant's `last_npy_data/` folder.**  
Bin size variants (30min/45min) — see final cell for instructions.


In [ ]:
import os, math, pickle, warnings, gc, logging
from collections import defaultdict

import numpy as np
import pandas as pd
from numpy.linalg import lstsq
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit
from tqdm import tqdm

# ── CONFIG (shared / fixed across all variants) ────────────────────────────────
# NOTE: Raw data is not publicly available (patient privacy). Set RAW_CSV to your local path.
RAW_CSV   = "path omitted for privacy — set to your local raw CSV path"
BASE_DIR  = "patient_level_split/sensitivity"
CHUNK_SIZE = 30000
MAX_SHIFT  = 672
PERIODS    = {'1h': 4, '1d': 96, '1w': 672}

os.makedirs(BASE_DIR, exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")

# ── Load & clean data (run once) ───────────────────────────────────────────────
STRUCTURAL     = ['이름', 'targetTime', 'idxInfo', 'dangerous_action']
LOCATION_FEATS = [
    'Daily_Entropy', 'Norm_Daily_Entropy', 'Location_Variability',
    'place_hallway', 'place_other', 'place_ward',
]
EMR_FEATS = [
    'DIG_3class', 'DIG_4class', 'DIG_withPsychosis', 'Restrictive_Interv',
    'age', 'sex', 'day_of_week', 'holidays',
]
SENSOR_BASE_FEATS = [
    'ENMO_max', 'ENMO_mean', 'ENMO_median', 'ENMO_min', 'ENMO_nunique', 'ENMO_std',
    'HR_max', 'HR_mean', 'HR_median', 'HR_min', 'HR_nunique', 'HR_std',
    'CALORIES_delta', 'DISTANCE_delta', 'SLEEP_delta', 'STEP_delta',
    'VE_tx', 'VE_tx_inj_time', 'nonwearing',
]
KEEP = STRUCTURAL + LOCATION_FEATS + EMR_FEATS + SENSOR_BASE_FEATS
EXCL = ['key','targetTime','idxInfo','hospital']
TH   = 20

data = pd.read_csv(RAW_CSV)
data = data[[c for c in KEEP if c in data.columns]].copy()
sensor_cols = [c for c in SENSOR_BASE_FEATS if c in data.columns]
data.loc[data['nonwearing'].astype(bool), sensor_cols] = 0
data = data.fillna(0)
data.rename(columns={'이름': 'key'}, inplace=True)
data['hospital'] = data['idxInfo'].str.split(pat='-', n=1).str[0]

categorical_columns = [
    c for c in data.columns
    if c not in EXCL and (data[c].nunique() <= TH or data[c].dtype == 'object')
]
continuous_columns = [
    c for c in data.columns
    if c not in EXCL and c not in categorical_columns and data[c].dtype in ['int64','float64']
]
hospitals = data['hospital'].unique()
print(f"Loaded {len(data)} rows, {len(hospitals)} hospitals: {hospitals}")
print(f"Categorical: {len(categorical_columns)}, Continuous: {len(continuous_columns)}")


In [ ]:
# ── Helper classes ─────────────────────────────────────────────────────────────

class DataAugmentor:
    def __init__(self, sigma=0.5, num_augments=5):
        self.sigma = sigma
        self.num_augments = num_augments

    def time_warping(self, data):
        return data * np.random.normal(1, self.sigma, size=data.shape)

    def augment_patient_data(self, df, target_columns, categorical_columns):
        if self.num_augments == 0:
            return df
        positive_data = df[df['dangerous_action'] == 1]
        augmented, pid = [], max(
            [int(k.split('_')[-1]) for k in df['key'].unique() if '_' in k] + [0]
        ) + 1
        for name, group in positive_data.groupby("key"):
            for _ in range(self.num_augments):
                np_data = group.copy()
                np_data["key"] = f"NewPatient_{pid}"; pid += 1
                for col in target_columns:
                    np_data[col] = self.time_warping(np_data[col].to_numpy())
                for col in group.columns:
                    if col in categorical_columns: continue
                    if pd.api.types.is_numeric_dtype(group[col]) and col not in target_columns:
                        np_data[col] += np.random.normal(0, self.sigma, size=np_data[col].shape)
                augmented.append(np_data)
        if not augmented:
            return df
        return pd.concat([df, pd.concat(augmented, ignore_index=True)], ignore_index=True)


class CosinorAnalyzer:
    def __init__(self, data, window_length_hours, coverage_threshold=0.5,
                 sensor='ENMO', name_col='이름', time_col='targetTime'):
        self.data = data.copy()
        self.window_length_hours = window_length_hours
        self.coverage_threshold  = coverage_threshold
        self.period  = 24.0
        self.sensor  = sensor
        self.name_col, self.time_col = name_col, time_col
        self.sensor_col = f"{sensor}_mean"
        if not pd.api.types.is_datetime64_any_dtype(self.data[time_col]):
            raise ValueError(f"'{time_col}' must be datetime.")
        self.data.sort_values([name_col, time_col], inplace=True)
        self.data.reset_index(drop=True, inplace=True)
        diffs = self.data.groupby(name_col)[time_col].diff().dropna()
        self.sampling_interval_minutes = diffs.median().total_seconds() / 60.0

    def _prepare_window(self, grp, ref, gref):
        start = ref - pd.Timedelta(hours=self.window_length_hours)
        win   = grp[(grp[self.time_col] >= start) & (grp[self.time_col] <= ref)]
        win   = win[win['nonwearing'] == False]
        expected = int((self.window_length_hours * 60) / self.sampling_interval_minutes)
        if len(win) == 0 or (len(win) / expected) < self.coverage_threshold:
            return None
        dfw = win.copy()
        dfw['t_hours'] = (dfw[self.time_col] - gref).dt.total_seconds() / 3600.0
        return dfw

    def _cosinor(self, dfw):
        y, t  = dfw[self.sensor_col].values, dfw['t_hours'].values
        omega = 2 * np.pi / self.period
        X     = np.column_stack([np.ones_like(t), np.cos(omega*t), np.sin(omega*t)])
        try:
            params, _, _, _ = lstsq(X, y, rcond=None)
        except Exception:
            return None
        M, A, B   = params
        amplitude = np.sqrt(A**2 + B**2)
        phase_hrs = (math.atan2(B, A) / (2*np.pi)) * self.period
        if phase_hrs < 0: phase_hrs += self.period
        return M, amplitude, phase_hrs

    def run(self):
        out = []
        for name, grp in tqdm(self.data.groupby(self.name_col), desc=f"Cosinor {self.sensor} {self.window_length_hours}h", leave=False):
            gref  = grp[self.time_col].min()
            start = gref + pd.Timedelta(hours=self.window_length_hours)
            for ref in grp[grp[self.time_col] >= start][self.time_col]:
                dfw = self._prepare_window(grp, ref, gref)
                res = None if dfw is None else self._cosinor(dfw)
                if res is None:
                    out.append({self.name_col: name, self.time_col: ref,
                                'MESOR': np.nan, 'Amplitude': np.nan, 'Phase_hours': np.nan})
                else:
                    M, amp, ph = res
                    out.append({self.name_col: name, self.time_col: ref,
                                'MESOR': M, 'Amplitude': amp, 'Phase_hours': ph})
        return pd.DataFrame(out)


# ── Pipeline functions ─────────────────────────────────────────────────────────

def PipelineSensor(df, short_win=48, long_win=168):
    df['targetTime'] = pd.to_datetime(df['targetTime'])
    enmo_s = CosinorAnalyzer(df, short_win, 0.5, 'ENMO').run()
    enmo_l = CosinorAnalyzer(df, long_win,  0.5, 'ENMO').run()
    hr_s   = CosinorAnalyzer(df, short_win, 0.5, 'HR').run()
    hr_l   = CosinorAnalyzer(df, long_win,  0.5, 'HR').run()
    em = pd.merge(enmo_s, enmo_l, on=['이름','targetTime'], suffixes=(f'_ENMO_{short_win}h',  '_ENMO_week'))
    hm = pd.merge(hr_s,   hr_l,   on=['이름','targetTime'], suffixes=(f'_HR_{short_win}h', '_HR_week'))
    return pd.merge(df, pd.merge(em, hm, on=['이름','targetTime']), on=['이름','targetTime'])


def create_binary_targets(df, target_col, periods):
    for p, steps in periods.items():
        df[f"{target_col}_{p}"] = (
            df.groupby('key')[target_col]
              .transform(lambda x: x.rolling(window=steps, min_periods=1).max())
              .shift(-steps).fillna(0).astype(int)
        )
    return df


def patient_split(df, group='key', random_state=42):
    rng  = np.random.RandomState(random_state)
    hosp = df['hospital'].iat[0]
    counts   = df.groupby(group).size().reset_index(name='total_records')
    ever_pos = df.groupby(group)['dangerous_action'].max().reset_index(name='ever_positive')
    counts   = counts.merge(ever_pos, on=group)
    pos_df, neg_df = counts[counts['ever_positive']==1], counts[counts['ever_positive']==0]

    if hosp == "동국대":
        labels  = df['dangerous_action'].values
        keys    = df.index.values
        sss1    = StratifiedShuffleSplit(n_splits=1, train_size=0.7, test_size=0.3, random_state=random_state)
        tri, tmpi = next(sss1.split(keys, labels))
        tr_raw, temp = df.iloc[tri].reset_index(drop=True), df.iloc[tmpi].reset_index(drop=True)
        sss2    = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=random_state)
        vi, tei = next(sss2.split(temp.index.values, temp['dangerous_action'].values))
        return tr_raw, temp.iloc[vi].reset_index(drop=True), temp.iloc[tei].reset_index(drop=True)

    elif hosp == "서울대병원":
        pos_keys = pos_df[group].tolist(); rng.shuffle(pos_keys)
        assert len(pos_keys)==14, f"SNU: expected 14 positives, got {len(pos_keys)}"
        tr_pos, val_pos, te_pos = pos_keys[:5], pos_keys[5:7], pos_keys[7:]
    elif hosp == "용인관리자":
        pos_keys = pos_df[group].tolist(); rng.shuffle(pos_keys)
        assert len(pos_keys)==16, f"Yongin: expected 16 positives, got {len(pos_keys)}"
        tr_pos, val_pos, te_pos = pos_keys[:8], pos_keys[10:12], pos_keys[12:]
    else:
        pos_keys = pos_df[group].tolist(); rng.shuffle(pos_keys)
        n = len(pos_keys)
        tr_pos  = pos_keys[:int(0.7*n)]
        val_pos = pos_keys[int(0.7*n):int(0.85*n)]
        te_pos  = pos_keys[int(0.85*n):]

    neg_keys = neg_df[group].tolist(); rng.shuffle(neg_keys)
    n_neg = len(neg_keys)
    n_tr  = int(np.floor(0.70 * n_neg))
    n_val = int(np.floor(0.15 * n_neg))
    tr_neg, val_neg, te_neg = neg_keys[:n_tr], neg_keys[n_tr:n_tr+n_val], neg_keys[n_tr+n_val:]
    tr  = df[df[group].isin(tr_pos  + tr_neg )].reset_index(drop=True)
    val = df[df[group].isin(val_pos + val_neg)].reset_index(drop=True)
    te  = df[df[group].isin(te_pos  + te_neg )].reset_index(drop=True)
    return tr, val, te


def process_key(key, df, seq_len, feats, targs, static_feats, max_shift):
    dfk = df[df['key']==key].sort_values('targetTime')
    if len(dfk) < seq_len + max_shift:
        return []
    X, Y   = dfk[feats].values, dfk[targs].values
    static = dfk[static_feats].iloc[0].values
    return [(static, X[i:i+seq_len], Y[i+seq_len]) for i in range(len(dfk)-seq_len-max_shift)]


def gen_sequences(df, seq_len, feats, targs, static_feats, max_shift=672):
    seqs = []
    for k in tqdm(df['key'].unique(), desc="Sequences", leave=False):
        seqs += process_key(k, df, seq_len, feats, targs, static_feats, max_shift)
    return seqs

print("Helper classes and functions defined.")


In [ ]:
# ── Master pipeline function ───────────────────────────────────────────────────

def resample_to_bin(df, bin_minutes):
    """Resample per-patient 15-min data to bin_minutes resolution."""
    freq, id_cols = f'{bin_minutes}T', ['key', 'hospital', 'idxInfo']
    binary_cols   = ['dangerous_action', 'nonwearing']
    parts = []
    for _, grp in df.groupby('key', sort=False):
        grp = grp.copy()
        grp['targetTime'] = pd.to_datetime(grp['targetTime'])
        grp = grp.set_index('targetTime').sort_index()
        id_vals  = {c: grp[c].iloc[0] for c in id_cols if c in grp.columns}
        grp_work = grp.drop(columns=[c for c in id_cols if c in grp.columns], errors='ignore')
        agg = {col: ('max' if col in binary_cols else
                     ('first' if grp_work[col].dtype == object else 'mean'))
               for col in grp_work.columns}
        rs = grp_work.resample(freq).agg(agg).dropna(how='all').fillna(0)
        for c, v in id_vals.items(): rs[c] = v
        parts.append(rs.reset_index())
    return pd.concat(parts, ignore_index=True)


def run_pipeline(sigma=0.5, cosinor_short=48, cosinor_long=168,
                 seq_len=192, bin_minutes=15, out_tag='baseline'):
    """
    Run the full sequence-creation pipeline with the given config.

    Parameters
    ----------
    sigma         : float  — DataAugmentor noise level (0.25 / 0.5 / 1.0)
    cosinor_short : int    — short cosinor window in hours (24 / 48 / 72)
    cosinor_long  : int    — long  cosinor window in hours (120 / 168 / 192)
    seq_len       : int    — input sequence length in timesteps (96 / 192 / 288)
    out_tag       : str    — subfolder name under patient_level_split/sensitivity/

    Outputs
    -------
    Pickles saved to: patient_level_split/sensitivity/<out_tag>/processed_sequences/
    """
    local_periods   = {
        '1h': max(1, round(60       / bin_minutes)),
        '1d': round(24 * 60         / bin_minutes),
        '1w': round(7 * 24 * 60     / bin_minutes),
    }
    local_seq_len   = round(48 * 60  / bin_minutes) if bin_minutes != 15 else seq_len
    local_max_shift = round(7 * 24 * 60 / bin_minutes)
    out_dir = os.path.join(BASE_DIR, out_tag, "processed_sequences")
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n{'='*65}")
    print(f" CONFIG: bin={bin_minutes}min, sigma={sigma}, cosinor={cosinor_short}h/{cosinor_long}h")
    print(f" seq_len={local_seq_len}, periods={local_periods}")
    print(f" Output: {out_dir}")
    print(f"{'='*65}")
    if bin_minutes != 15:
        print(f"Resampling 15-min → {bin_minutes}-min bins...")
        work_data = resample_to_bin(data, bin_minutes)
        print(f"  {len(data)} rows → {len(work_data)} rows")
    else:
        work_data = data

    drop_cols = [
        f'Amplitude_HR_{cosinor_short}h', f'Amplitude_ENMO_{cosinor_short}h',
        f'MESOR_HR_{cosinor_short}h',     f'MESOR_ENMO_{cosinor_short}h',
        f'Phase_hours_HR_{cosinor_short}h', f'Phase_hours_ENMO_{cosinor_short}h',
        'Normalized_Eight_Hour_Entropy', 'Eight_Hour_Entropy',
    ]

    def full_pipe(df):
        d2 = PipelineSensor(df.rename(columns={'key':'이름'}),
                            short_win=cosinor_short, long_win=cosinor_long)
        d2.rename(columns={'이름':'key'}, inplace=True)
        for col in drop_cols:
            d2.drop(columns=[col], inplace=True, errors=True)
        return create_binary_targets(d2, 'dangerous_action', local_periods).fillna(0)

    for hosp in work_data['hospital'].unique():
        print(f"\n--- {hosp} ---")
        dfh = work_data[work_data['hospital']==hosp].copy()

        # Patient-level split
        tr_raw, val_raw, te_raw = patient_split(dfh)

        # Label-encode
        encs = {}
        for c in categorical_columns:
            if c == 'idxInfo': continue
            le = LabelEncoder().fit(dfh[c].astype(str))
            encs[c] = le
            for d in (tr_raw, val_raw, te_raw):
                d[c] = le.transform(d[c].astype(str))

        # Augment train
        if hosp == "동국대":
            pos_df_h = tr_raw[tr_raw['dangerous_action']==1]
            neg_df_h = tr_raw[tr_raw['dangerous_action']==0]
            pos_aug  = DataAugmentor(sigma=sigma, num_augments=20).augment_patient_data(
                pos_df_h, continuous_columns, categorical_columns)
            neg_work = neg_df_h.copy(); neg_work['dangerous_action'] = 1
            neg_aug  = DataAugmentor(sigma=sigma, num_augments=20).augment_patient_data(
                neg_work, continuous_columns, categorical_columns)
            neg_aug['dangerous_action'] = 0
            train_aug = pd.concat([pos_aug, neg_aug], ignore_index=True)
        elif hosp == "서울대병원":
            train_aug = DataAugmentor(sigma=sigma, num_augments=5).augment_patient_data(
                tr_raw, continuous_columns, categorical_columns)
        else:
            train_aug = DataAugmentor(sigma=sigma, num_augments=6).augment_patient_data(
                tr_raw, continuous_columns, categorical_columns)

        tr_proc  = full_pipe(train_aug)
        val_proc = full_pipe(val_raw)
        te_proc  = full_pipe(te_raw)

        ALL_FEATS    = [c for c in tr_proc.columns
                        if c not in ['key','targetTime','dangerous_action','idxInfo','hospital']
                        + [f"dangerous_action_{p}" for p in local_periods]]
        static_feats = [c for c in ALL_FEATS if c in categorical_columns]
        seq_feats    = [c for c in ALL_FEATS if c not in static_feats]
        targets      = [f"dangerous_action_{p}" for p in local_periods]

        for name, df_split in (('train',tr_proc),('val',val_proc),('test',te_proc)):
            seqs = gen_sequences(df_split, seq_len, seq_feats, targets, static_feats)
            if not seqs:
                print(f"  SKIP {hosp}_{name}: 0 sequences")
                continue
            nchunks = math.ceil(len(seqs) / CHUNK_SIZE)
            for i in range(nchunks):
                chunk = seqs[i*CHUNK_SIZE:(i+1)*CHUNK_SIZE]
                fname = f"{hosp}_{name}_chunk_{i}.pkl"
                with open(os.path.join(out_dir, fname), 'wb') as f:
                    pickle.dump(chunk, f)
                print(f"  → {fname} ({len(chunk)} seqs)")
        gc.collect()

    print(f"\nDone. Pickles in: {out_dir}")
    return out_dir

print("run_pipeline() defined.")


In [ ]:
# ── Memmap creation function ──────────────────────────────────────────────────

def filter_static(s):
    arr = np.array(s, dtype=object)
    filt = [v for v in arr if not isinstance(v, str)]
    return np.array(filt, dtype=np.float32) if filt else None

def clean_seq(x):
    if not hasattr(x,"__len__") or len(x)==0: return x
    bad = {j for j,v in enumerate(x[0]) if isinstance(v, str)}
    if not bad: return x
    return [[v for j,v in enumerate(row) if j not in bad] for row in x]

def write_mm(path, n, shape, dtype=np.float32):
    mm = np.memmap(path, mode='w+', dtype=dtype, shape=(n,)+shape)
    return mm, 0

def push(mm, ptr, batch):
    n = batch.shape[0]; mm[ptr:ptr+n] = batch; return ptr+n


def create_memmaps(seqs_dir, root_dir):
    """
    Convert pickle chunks in seqs_dir into memory-mapped .npy arrays in root_dir.
    Structure:  root_dir/HospitalsData/<hosp>/{sequence,static,targets}_{train,val,test}.npy
                root_dir/GlobalData/{sequence_data,static_data,targets}.npy
    """
    global_dir = os.path.join(root_dir, "GlobalData")
    hosp_dir   = os.path.join(root_dir, "HospitalsData")
    os.makedirs(global_dir, exist_ok=True)
    os.makedirs(hosp_dir,   exist_ok=True)
    CHUNK = 100

    print(f"\nCreating memmaps in: {root_dir}")

    # Group files by hospital + split
    hfiles = defaultdict(lambda: {"train":[],"val":[],"test":[]})
    for fn in os.listdir(seqs_dir):
        if not fn.endswith(".pkl") or "_chunk_" not in fn: continue
        head = fn.split("_chunk_")[0]
        if   head.endswith("_train"): split="train"; hosp=head[:-6]
        elif head.endswith("_val"):   split="val";   hosp=head[:-4]
        elif head.endswith("_test"):  split="test";  hosp=head[:-5]
        else: continue
        hfiles[hosp][split].append(os.path.join(seqs_dir, fn))
    for hosp, splits in hfiles.items():
        for sp in ("train","val","test"):
            splits[sp].sort(key=lambda p: int(p.split("_chunk_")[-1].split(".")[0]))

    # Determine sample shapes from first file
    some = next(iter(hfiles.values()))
    first_file = (some["train"] or some["val"] or some["test"])[0]
    with open(first_file, "rb") as f:
        s0, x0, y0 = pickle.load(f)[0]
    fs0 = filter_static(s0); rx0 = clean_seq(x0)
    static_dim = fs0.shape[0]
    seq_shape  = np.asarray(rx0, dtype=np.float32).shape
    tgt_dim    = np.asarray(y0,  dtype=np.float32).shape[0]
    print(f"  Shapes → static:{static_dim}, seq:{seq_shape}, targets:{tgt_dim}")

    # Count samples per hospital/split
    hinfo = {}
    for hosp, splits in hfiles.items():
        info = {}
        for sp in ("train","val"):
            n = 0
            for pkl in splits[sp]:
                with open(pkl,"rb") as f:
                    seqs = pickle.load(f)
                for s,x,y in seqs:
                    if filter_static(s) is None: continue
                    try: np.asarray(clean_seq(x),dtype=np.float32); n+=1
                    except: pass
                del seqs; gc.collect()
            info[f"{sp}_count"] = n
        te_n = 0
        for pkl in splits["test"]:
            with open(pkl,"rb") as f:
                seqs = pickle.load(f)
            for s,x,y in seqs:
                if filter_static(s) is None: continue
                try: np.asarray(clean_seq(x),dtype=np.float32); te_n+=1
                except: pass
            del seqs; gc.collect()
        info["test_count"] = te_n
        hinfo[hosp] = info
        logging.info(f"{hosp}: train={info['train_count']}, val={info['val_count']}, test={te_n}")

    total_global = sum(v["test_count"] for v in hinfo.values())

    # Create global memmaps
    g_sm, g_sp = write_mm(os.path.join(global_dir,"static_data.npy"),   total_global, (static_dim,))
    g_xm, g_xp = write_mm(os.path.join(global_dir,"sequence_data.npy"), total_global, seq_shape)
    g_ym, g_yp = write_mm(os.path.join(global_dir,"targets.npy"),       total_global, (tgt_dim,))

    # Populate
    for hosp, splits in hfiles.items():
        info  = hinfo[hosp]
        odir  = os.path.join(hosp_dir, hosp)
        os.makedirs(odir, exist_ok=True)

        for sp, mm_key in (("train","train"),("val","val"),("test","test")):
            n = info[f"{sp}_count"]
            if n == 0: continue
            sm, sp_ = write_mm(os.path.join(odir,f"static_{sp}.npy"),   n, (static_dim,))
            xm, xp  = write_mm(os.path.join(odir,f"sequence_{sp}.npy"), n, seq_shape)
            ym, yp  = write_mm(os.path.join(odir,f"targets_{sp}.npy"),  n, (tgt_dim,))

            buf_s, buf_x, buf_y = [], [], []
            for pkl in splits[sp]:
                with open(pkl,"rb") as f:
                    seqs = pickle.load(f)
                for s,x,y in seqs:
                    fs = filter_static(s)
                    if fs is None: continue
                    rx = clean_seq(x)
                    try:
                        ax = np.asarray(rx,dtype=np.float32)
                        ay = np.asarray(y, dtype=np.float32)
                    except: continue
                    buf_s.append(fs); buf_x.append(ax); buf_y.append(ay)
                    if len(buf_s) >= CHUNK:
                        sp_ = push(sm, sp_, np.stack(buf_s))
                        xp  = push(xm, xp,  np.stack(buf_x))
                        yp  = push(ym, yp,  np.stack(buf_y))
                        # also write test → global
                        if sp == "test":
                            g_sp = push(g_sm, g_sp, np.stack(buf_s))
                            g_xp = push(g_xm, g_xp, np.stack(buf_x))
                            g_yp = push(g_ym, g_yp, np.stack(buf_y))
                        buf_s.clear(); buf_x.clear(); buf_y.clear()
                del seqs; gc.collect()
            if buf_s:
                sp_ = push(sm, sp_, np.stack(buf_s))
                xp  = push(xm, xp,  np.stack(buf_x))
                yp  = push(ym, yp,  np.stack(buf_y))
                if sp == "test":
                    g_sp = push(g_sm, g_sp, np.stack(buf_s))
                    g_xp = push(g_xm, g_xp, np.stack(buf_x))
                    g_yp = push(g_ym, g_yp, np.stack(buf_y))
                buf_s.clear(); buf_x.clear(); buf_y.clear()
            sm.flush(); xm.flush(); ym.flush()
            logging.info(f"  {hosp}/{sp}: {sp_} samples written")

    g_sm.flush(); g_xm.flush(); g_ym.flush()
    print(f"Done. Global test: {g_sp} samples in {global_dir}")
    return root_dir

print("create_memmaps() defined.")


---
## Baseline Pipeline (Default)
σ=0.5, cosinor 48h/168h, seq_len=192. Run this first to reproduce the baseline results reported in the paper.

In [ ]:
seqs_dir = run_pipeline(sigma=0.5, cosinor_short=48, cosinor_long=168,
                        seq_len=192, out_tag='baseline')
root_dir = os.path.join(BASE_DIR, 'baseline', 'last_npy_data')
create_memmaps(seqs_dir, root_dir)
print(f"\n✓ Baseline complete → {root_dir}")


---
## Sensitivity: Aug σ = 0.25 (lower noise)
Augmentation sigma halved — tests whether the default σ=0.5 is unnecessarily large.

In [ ]:
seqs_dir = run_pipeline(sigma=0.25, cosinor_short=48, cosinor_long=168,
                        seq_len=192, out_tag='aug_sigma025')
seqs_dir = os.path.join(BASE_DIR, 'aug_sigma025', 'processed_sequences')
root_dir = os.path.join(BASE_DIR, 'aug_sigma025', 'last_npy_data')
create_memmaps(seqs_dir, root_dir)
print(f"\n✓ Sensitivity: Aug σ = 0.25 (lower noise) complete → {root_dir}")
print("Point FL training at this root_dir to run the sensitivity experiment.")


---
## Sensitivity: Aug σ = 1.0 (higher noise)
Augmentation sigma doubled — tests robustness to stronger perturbations.

In [ ]:
seqs_dir = run_pipeline(sigma=1.0, cosinor_short=48, cosinor_long=168,
                        seq_len=192, out_tag='aug_sigma100')
seqs_dir = os.path.join(BASE_DIR, 'aug_sigma100', 'processed_sequences')
root_dir = os.path.join(BASE_DIR, 'aug_sigma100', 'last_npy_data')
create_memmaps(seqs_dir, root_dir)
print(f"\n✓ Sensitivity: Aug σ = 1.0 (higher noise) complete → {root_dir}")
print("Point FL training at this root_dir to run the sensitivity experiment.")


---
## Sensitivity: Cosinor Window 24h / 120h (shorter)
Shorter circadian windows — tests whether 48h/168h is necessary or a 24h/120h window suffices.

In [ ]:
seqs_dir = run_pipeline(sigma=0.5, cosinor_short=24, cosinor_long=120,
                        seq_len=192, out_tag='cosinor_24_120')
seqs_dir = os.path.join(BASE_DIR, 'cosinor_24_120', 'processed_sequences')
root_dir = os.path.join(BASE_DIR, 'cosinor_24_120', 'last_npy_data')
create_memmaps(seqs_dir, root_dir)
print(f"\n✓ Sensitivity: Cosinor Window 24h / 120h (shorter) complete → {root_dir}")
print("Point FL training at this root_dir to run the sensitivity experiment.")


---
## Sensitivity: Cosinor Window 72h / 192h (longer)
Longer circadian windows — tests whether extending beyond 48h/168h improves circadian capture.

In [ ]:
seqs_dir = run_pipeline(sigma=0.5, cosinor_short=72, cosinor_long=192,
                        seq_len=192, out_tag='cosinor_72_192')
seqs_dir = os.path.join(BASE_DIR, 'cosinor_72_192', 'processed_sequences')
root_dir = os.path.join(BASE_DIR, 'cosinor_72_192', 'last_npy_data')
create_memmaps(seqs_dir, root_dir)
print(f"\n✓ Sensitivity: Cosinor Window 72h / 192h (longer) complete → {root_dir}")
print("Point FL training at this root_dir to run the sensitivity experiment.")


---
## Sensitivity: Sequence Length = 96 steps (24h input window)
Shorter input window (24h vs default 48h) — tests whether 2-day context is necessary.

In [ ]:
seqs_dir = run_pipeline(sigma=0.5, cosinor_short=48, cosinor_long=168,
                        seq_len=96, out_tag='seqlen_096')
seqs_dir = os.path.join(BASE_DIR, 'seqlen_096', 'processed_sequences')
root_dir = os.path.join(BASE_DIR, 'seqlen_096', 'last_npy_data')
create_memmaps(seqs_dir, root_dir)
print(f"\n✓ Sensitivity: Sequence Length = 96 steps (24h input window) complete → {root_dir}")
print("Point FL training at this root_dir to run the sensitivity experiment.")


---
## Sensitivity: Sequence Length = 288 steps (72h input window)
Longer input window (72h vs default 48h) — tests whether 3-day context improves predictions.

In [ ]:
seqs_dir = run_pipeline(sigma=0.5, cosinor_short=48, cosinor_long=168,
                        seq_len=288, out_tag='seqlen_288')
seqs_dir = os.path.join(BASE_DIR, 'seqlen_288', 'processed_sequences')
root_dir = os.path.join(BASE_DIR, 'seqlen_288', 'last_npy_data')
create_memmaps(seqs_dir, root_dir)
print(f"\n✓ Sensitivity: Sequence Length = 288 steps (72h input window) complete → {root_dir}")
print("Point FL training at this root_dir to run the sensitivity experiment.")


---
## Sensitivity: Bin Size (30-min and 45-min)
Tests robustness to temporal resolution. Data is resampled from 15-min baseline using pandas resample.

| Variant | Bin | seq_len | Periods (1h / 1d / 1w) |
|---|---|---|---|
| bin_030 | 30 min | 96 | 2 / 48 / 336 |
| bin_045 | 45 min | 64 | 1 / 32 / 224 |

**Note**: For bin_045, the 1h prediction horizon is 45 min (closest integer bin).


In [ ]:
# 30-min bins
seqs_dir = run_pipeline(sigma=0.5, cosinor_short=48, cosinor_long=168,
                        bin_minutes=30, out_tag='bin_030')
root_dir = os.path.join(BASE_DIR, 'bin_030', 'last_npy_data')
create_memmaps(seqs_dir, root_dir)
print(f'\n✓ bin_030 complete → {root_dir}')

# 45-min bins (note: 1h horizon ≈ 45 min)
seqs_dir = run_pipeline(sigma=0.5, cosinor_short=48, cosinor_long=168,
                        bin_minutes=45, out_tag='bin_045')
root_dir = os.path.join(BASE_DIR, 'bin_045', 'last_npy_data')
create_memmaps(seqs_dir, root_dir)
print(f'\n✓ bin_045 complete → {root_dir}')
